In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
caminho_base = Path(r"C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade")

pastas_anos = [pasta for pasta in caminho_base.iterdir()]

for ano in pastas_anos[:-2]:
    arquivos = list(ano.glob("*.csv"))
    for arquivo in arquivos:
        try:
            # modificando os arquivos em memória
            df = pd.read_csv(arquivo, sep=";", encoding="utf-8", engine="python", index_col=0)
            df_convertido = df.iloc[:, 2:].replace(',', '.', regex=True).astype(float)
            df_mascarado = df_convertido.mask(df_convertido == -9999.0)
            df_convertido.info()
            medias_coluna = df_mascarado.mean()
            df_convertido = df_mascarado.fillna(medias_coluna)
            df_final = pd.concat([df.iloc[:, :2], df_convertido], axis=1)

            df_final.to_csv(ano / arquivo.name, sep=';', encoding='utf-8', index=False)

        except Exception as e:
            display(e)


In [27]:
path = Path(r'C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade\2026')

arquivos = list(path.glob('*.csv'))
colunas_certas = ['Data', 'Hora UTC', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
       'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
       'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
       'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
       'RADIACAO GLOBAL (Kj/m²)',
       'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
       'TEMPERATURA DO PONTO DE ORVALHO (°C)',
       'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
       'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
       'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
       'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
       'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
       'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
       'UMIDADE RELATIVA DO AR, HORARIA (%)',
       'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, RAJADA MAXIMA (m/s)',
       'VENTO, VELOCIDADE HORARIA (m/s)']

for arquivo in arquivos:
    df = pd.read_csv(arquivo, sep=';', encoding='ISO-8859-1', engine='python', skiprows=8, usecols=colunas_certas)
    df = df.loc[df['Data'].isin(['2026/10/01', '2026/10/02', '2026/10/29', '2026/10/30'])]
    df_convertido = df.iloc[:, 2:].replace(',', '.', regex=True).astype(float)
    df_convertido = df_convertido.fillna(df_convertido.mean())
    df_final = pd.concat([df.iloc[:, :2], df_convertido], axis=1)

    df_final.to_csv(arquivo, sep=';', encoding='utf-8', index=False)

C:\Users\Caio\AppData\Local\Temp\ipykernel_4600\2425029699.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_convertido = df.iloc[:, 2:].replace(',', '.', regex=True).astype(float)
C:\Users\Caio\AppData\Local\Temp\ipykernel_4600\2425029699.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_convertido = df.iloc[:, 2:].replace(',', '.', regex=True).astype(float)


# Tabelas dia de Chuva

In [6]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd

# Caminhos
BASE_DIR = Path(r"C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade")
BASE_SQL = Path(r"C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\documentacao do grupo\Diagramas\DDL")
OUTPUT_FILE = BASE_DIR / "DIM_QT_CHUVA.csv"
SQL_FILE = BASE_SQL / "DIM_POSICAOGEOGRAFICA.sql"

# Datas das Eleições (1º e 2º Turno) e vésperas para cada ano
DATAS_ELEICAO = {
    2014: ["2014-10-04", "2014-10-05", "2014-10-25", "2014-10-26"],
    2018: ["2018-10-06", "2018-10-07", "2018-10-27", "2018-10-28"],
    2022: ["2022-10-01", "2022-10-02", "2022-10-29", "2022-10-30"],
    2026: ["2026-10-03", "2026-10-04", "2026-10-24", "2026-10-25"]
}

# Parâmetros da variação ponderada ao redor da média
SEED = 3                  # Garante reprodutibilidade dos números gerados
DESVIO_CHUVA_PCT = 0.37    # Desvio padrão de 12% em relação à média da chuva
DESVIO_UMIDADE_PCT = 0.25  # Desvio padrão de 4% em relação à média da umidade


def parse_sql_enderecos(sql_path):
    """Extrai os endereços e anos diretamente do arquivo SQL."""
    enderecos = []
    with open(sql_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    matches = re.findall(r"\((\d{4}),\s*N'([^']+)'", content)
    for ano, endereco in matches:
        if int(ano) in DATAS_ELEICAO:
            enderecos.append({"ANO": int(ano), "DS_ENDERECO_LOCVT_ORIGINAL": endereco})

    return pd.DataFrame(enderecos).drop_duplicates()


def extrair_municipio_arquivo(nome_arquivo):
    """Extrai o nome do município a partir do nome do arquivo INMET."""
    partes = nome_arquivo.split("_")
    if len(partes) >= 5:
        return partes[4].upper()
    return ""


def processar_arquivo_inmet(caminho_arquivo, ano_subpasta):
    """Lê o arquivo INMET e filtra apenas D-1 e D0 da eleição correspondente."""
    municipio = extrair_municipio_arquivo(caminho_arquivo.name)
    encoding = "utf-8"
    skiprows = 0

    with open(caminho_arquivo, "r", encoding=encoding, errors="ignore") as f:
        for idx, line in enumerate(f):
            if idx > 25:
                break
            if re.search(r"(DATA|Data|DT_MEDICAO)", line, re.IGNORECASE):
                skiprows = idx
                break

    try:
        df = pd.read_csv(caminho_arquivo, sep=";", skiprows=skiprows, encoding=encoding, decimal=",", engine="python")
    except Exception:
        return None

    df.columns = [str(c).strip() for c in df.columns]

    col_data = next((c for c in df.columns if re.search(r"DATA|DT_MEDICAO", c, re.IGNORECASE)), None)
    col_chuva = next((c for c in df.columns if re.search(r"PRECIPITA", c, re.IGNORECASE)), None)
    col_umidade = next((c for c in df.columns if re.search(r"UMIDADE RELATIVA DO AR", c, re.IGNORECASE)), None)

    if not all([col_data, col_chuva, col_umidade]):
        return None

    df["DATA_FMT"] = pd.to_datetime(df[col_data].astype(str).str.replace("/", "-"), errors="coerce").dt.strftime("%Y-%m-%d")

    datas_alvo = DATAS_ELEICAO.get(int(ano_subpasta), [])
    df_filtrado = df[df["DATA_FMT"].isin(datas_alvo)].copy()

    if df_filtrado.empty:
        return None

    for col in [col_chuva, col_umidade]:
        series = df_filtrado[col]
        if series.dtype == object:
            series = series.astype(str).str.replace(",", ".")
        df_filtrado[col] = pd.to_numeric(series, errors="coerce")
        df_filtrado[col] = df_filtrado[col].apply(lambda x: None if (pd.notna(x) and x < -100) else x)

    return pd.DataFrame({
        "ANO": [int(ano_subpasta)],
        "MUNICIPIO": [municipio],
        "PRECIPITACAO_MEDIA": [df_filtrado[col_chuva].mean()],
        "UMIDADE_MEDIA": [df_filtrado[col_umidade].mean()]
    })


def aplicar_variacao_ponderada(media, desvio_pct, rng, minimo=0.0, maximo=None, manter_zero=False):
    """Gera um valor aleatório com distribuição normal orbitando a média."""
    if pd.isna(media):
        return np.nan
    if manter_zero and media <= 0:
        return 0.0

    desvio_abs = abs(media) * desvio_pct
    valor = rng.normal(loc=media, scale=desvio_abs)

    if minimo is not None:
        valor = max(minimo, valor)
    if maximo is not None:
        valor = min(maximo, valor)

    return valor


def gerar_dim_qtde_chuva():
    rng = np.random.default_rng(SEED)
    lista_dfs = []

    for subpasta in BASE_DIR.iterdir():
        if subpasta.is_dir() and subpasta.name.isdigit() and int(subpasta.name) in DATAS_ELEICAO:
            ano = subpasta.name
            print(f"Processando {ano}...")
            for arquivo in subpasta.glob("*.[cC][sS][vV]"):
                df_proc = processar_arquivo_inmet(arquivo, ano)
                if df_proc is not None and not df_proc.empty:
                    lista_dfs.append(df_proc)

    if not lista_dfs:
        print("Nenhum arquivo processado.")
        return

    df_clima = pd.concat(lista_dfs, ignore_index=True)

    if not SQL_FILE.exists():
        print(f"Arquivo SQL não encontrado: {SQL_FILE}")
        return

    df_locais = parse_sql_enderecos(SQL_FILE)

    def associar_clima(row):
        ano = row["ANO"]
        endereco = str(row["DS_ENDERECO_LOCVT_ORIGINAL"]).upper()

        clima_ano = df_clima[df_clima["ANO"] == ano]
        if clima_ano.empty:
            return pd.Series([np.nan, np.nan])

        media_precip = None
        media_umid = None

        # Busca o município no endereço
        for _, clima_row in clima_ano.iterrows():
            if clima_row["MUNICIPIO"] and clima_row["MUNICIPIO"] in endereco:
                media_precip = clima_row["PRECIPITACAO_MEDIA"]
                media_umid = clima_row["UMIDADE_MEDIA"]
                break

        # Fallback: média estadual do ano
        if media_precip is None:
            media_precip = clima_ano["PRECIPITACAO_MEDIA"].mean()
            media_umid = clima_ano["UMIDADE_MEDIA"].mean()

        # Aplica variação ponderada (orbita a média do município/estado)
        precip_final = aplicar_variacao_ponderada(
            media_precip, DESVIO_CHUVA_PCT, rng, minimo=0.0, manter_zero=True
        )
        umid_final = aplicar_variacao_ponderada(
            media_umid, DESVIO_UMIDADE_PCT, rng, minimo=0.0, maximo=100.0
        )

        return pd.Series([precip_final, umid_final])

    df_locais[["PRECIPITACAO_TOTAL", "UMIDADE_RELATIVA_AR"]] = df_locais.apply(associar_clima, axis=1)

    df_locais["PRECIPITACAO_TOTAL"] = df_locais["PRECIPITACAO_TOTAL"].round(2)
    df_locais["UMIDADE_RELATIVA_AR"] = df_locais["UMIDADE_RELATIVA_AR"].round(2)

    df_locais = df_locais[["DS_ENDERECO_LOCVT_ORIGINAL", "ANO", "PRECIPITACAO_TOTAL", "UMIDADE_RELATIVA_AR"]]
    df_locais.to_csv(OUTPUT_FILE, index=False, sep=";", encoding="utf-8-sig")
    print(f"Tabela gerada com sucesso: {OUTPUT_FILE}")


if __name__ == "__main__":
    gerar_dim_qtde_chuva()

Processando 2014...
Processando 2018...
Processando 2022...
Processando 2026...
Tabela gerada com sucesso: C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade\DIM_QT_CHUVA.csv


In [33]:
ARQUIVO_CSV = Path(
    r'C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade\Quanto Choveu\DIM_QTDE_CHUVA.csv'
)


df = pd.read_csv(ARQUIVO_CSV, sep=',', engine='python')

df['CHOVEU_FORTE'] = (df['PRECIPITAÇÃO_TOTAL_HORA'] > 1)

display(df['CHOVEU_FORTE'].value_counts())

df

# df_ordenado = df.sort_values(["ANO", "DIA", "HORA"], ascending=[True, True, True])

# df_ordenado.to_csv(ARQUIVO_CSV, index=False, encoding='utf-8-sig')

CHOVEU_FORTE
False    270
True      18
Name: count, dtype: int64

,DIA,ANO,HORA,PRECIPITAÇÃO_TOTAL_HORA,UMIDADE_RELATIVA_AR,CHOVEU_FORTE
0,4,2014,0,0.000000,83.176471,False
1,4,2014,1,0.294118,85.705882,False
2,4,2014,2,1.929412,87.352941,True
3,4,2014,3,0.364706,87.941176,False
4,4,2014,4,0.305882,88.058824,False
...,...,...,...,...,...,...
283,30,2022,19,0.000000,58.600000,False
284,30,2022,20,0.000000,59.466667,False
285,30,2022,21,0.000000,63.866667,False
286,30,2022,22,0.000000,72.266667,False


In [12]:
df = pd.read_csv(r"C:\Users\Caio\Desktop\Impacta - Obsidian\IMPACTA\4° Semestre 2026 Impacta\MATERIAIS\Inovation Lab\DataMind-Analytics\dados\Pluviosidade\Quanto Choveu\DIM_QT_CHUVA.csv", sep=';', engine='python', encoding='utf-8')


df.drop_duplicates(['DS_ENDERECO_LOCVT_ORIGINAL', 'ANO'])

df['DS_ENDERECO_LOCVT_ORIGINAL'].value_counts()


DS_ENDERECO_LOCVT_ORIGINAL
AV 21 DE ABRIL 110                                                  4
ACESSO PELO KM 26 DA ESTRADA MANOEL URBANO - RAMAL DO 26 - KM 03    4
AV ADALBERTO VALE, 58 (BECO SALUSTIANO)                             4
AV FRANCISCO FERREIRA DE SOUZA                                      4
AV GAL RODRIGO OTAVIO JORDAO RAMOS 1600                             4
                                                                   ..
RUA SÃO CRISTOVÃO, S/N                                              1
RUA SÃO JUDAS TADEU (PRIMEIRA RUA), S/N                             1
RUA PADRE JOAO HOSPEL                                               1
RUA PEDRO ALVARES CABRAL, S/N, - DIST. DO SUCUNDURI                 1
RUA PEDRO DIAS LEME, 203                                            1
Name: count, Length: 3083, dtype: int64